# Classificação


In [16]:
%%html
<link rel="stylesheet" href="./style.css">


In [17]:
import import_ipynb  # noqa: F401
from treatment import (  # type: ignore
    t_df,
)

In [18]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from illustration import (
    best_parameters_table,
    class_distribution_table,
    fold_size_table,
    grid_search_results_table,
    stratified_fold_table,
    train_test_split_table,
)

seed = 24

## Separação dos dados

Separamos os dados em **treino** (`80%`) e **teste** (`20%`) de forma estratificada pelo atributo objetivo:

- **Grau de risco**: `low`, `mid`, `high`.


In [19]:
input_data = t_df.drop(columns=["risk_level"])
output_data = t_df["risk_level"]

(
    input_data_for_train,
    input_data_for_test,
    output_data_for_train,
    output_data_for_test,
) = train_test_split(
    input_data,
    output_data,
    test_size=0.2,
    random_state=seed,
    stratify=output_data,
)

In [20]:
display(
    train_test_split_table(
        input_data_for_train,
        input_data_for_test,
    )
)

,dataset,samples,percentage,features
0,train,66,79.52%,5
1,test,17,20.48%,5


## Balanceamento


In [21]:
display(
    class_distribution_table(
        output_data_for_train, caption="Distribuição das classes para o treino"
    )
)

display(
    class_distribution_table(
        output_data_for_test, caption="Distribuição das classes para o teste"
    )
)


,count,percentage
Risk level,,
low,28,42.42%
mid,21,31.82%
high,17,25.76%


,count,percentage
Risk level,,
low,8,47.06%
mid,5,29.41%
high,4,23.53%


Dado que o desbalanceamento entre as classes não é acentuado, decidimos não fazer um balanceamento artificial por ora, e observar o desempenho das técnicas de ajuste de hiper-parâmetros.


## KNN

Usamos a padronização `StandardScaler` para viabilizar os cálculos de distâncias.


In [22]:
knn_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier()),
    ]
)

### Ajuste de hiper-parâmetros

Utilizamos o **GridSearch** para encontrarmos os melhores valores de hiper-parâmetros.

- Testamos os seguintes valores de **K**-vizinhos: [`3`, `5`, `7`, `9`, `11`].
- Testamos os seguintes métodos de cálculo de **pesos**: [`uniform`, `distance`].
- Testamos as seguintes métricas de **distância**: [`euclidean`, `manhattan`].

Como **critério de seleção**, utilizamos `f1_macro`.

- A métrica `F1` que combina _precision_ e _recall_ para cada classe objetivo.
- O `F1 macro` calcula a média dos valores de F1 das três.

Executamos o GridSearch em `5` **folds** estratificados no conjunto de `treino`.


In [23]:
stratified_fold_table(
    input_data_for_train,
    output_data_for_train,
)

,fold,validation_samples,high_samples,low_samples,mid_samples
0,1,14,4,6,4
1,2,13,3,6,4
2,3,13,3,6,4
3,4,13,3,5,5
4,5,13,4,5,4


In [24]:
fold_size_table(
    input_data_for_train,
    output_data_for_train,
)

,fold,training_samples,validation_samples,total_samples
0,1,52,14,66
1,2,53,13,66
2,3,53,13,66
3,4,53,13,66
4,5,53,13,66


In [25]:
knn_param_grid = {
    "classifier__n_neighbors": [3, 5, 7, 9, 11],
    "classifier__weights": ["uniform", "distance"],
    "classifier__metric": ["euclidean", "manhattan"],
}

stratified_k_fold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=seed,
)

knn_grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_param_grid,
    scoring="f1_macro",
    cv=stratified_k_fold,
    n_jobs=-1,
)


In [26]:
knn_grid_search.fit(input_data_for_train, output_data_for_train)
display(grid_search_results_table(knn_grid_search))
display(best_parameters_table(knn_grid_search))


,rank,metric,n_neighbors,weights,mean_f1_macro,std_f1_macro
1,1,euclidean,3,distance,0.6125,0.0819
6,2,euclidean,9,uniform,0.6047,0.0476
8,3,euclidean,11,uniform,0.5976,0.1082
4,4,euclidean,7,uniform,0.5899,0.1084
7,5,euclidean,9,distance,0.5870,0.0786
11,6,manhattan,3,distance,0.5726,0.0981
0,7,euclidean,3,uniform,0.5707,0.0855
9,8,euclidean,11,distance,0.5693,0.0842
5,9,euclidean,7,distance,0.5623,0.1208
12,10,manhattan,5,uniform,0.5618,0.1676


,parameter,value
0,classifier__metric,euclidean
1,classifier__n_neighbors,3
2,classifier__weights,distance
3,best_f1_macro,0.6125


## Decision tree


In [27]:
decision_tree_pipeline = Pipeline(
    [
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=seed,
            ),
        ),
    ]
)

### Ajuste de hiper-parâmetros

Utilizamos o **GridSearch** para encontrarmos os melhores valores de hiper-parâmetros.

- Testamos os seguintes **critérios** de decisão: [`gini`, `entropy`].
- Testamos as seguintes **profundidades** máximas: [`3`, `5`, `7`, `10`, `15`, `None`].
- Testamos as seguintes quantidades mínimas de **amostras** para divisão de nó: [`2`, `5`, `10`].
- Testamos as seguintes quantidades mínimas de **amostras** que um nó folha deve ter: [`1`, `2`, `5`].

Como **critério de seleção**, utilizamos `f1_macro`.

Executamos o GridSearch em `5` **folds** estratificados no conjunto de `treino`.


In [28]:
tree_param_grid = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [3, 5, 7, 10, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 5],
}

tree_grid_search = GridSearchCV(
    estimator=decision_tree_pipeline,
    param_grid=tree_param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
)

In [30]:
tree_grid_search.fit(input_data_for_train, output_data_for_train)
display(grid_search_results_table(tree_grid_search))
display(best_parameters_table(tree_grid_search))

,rank,criterion,max_depth,min_samples_leaf,min_samples_split,mean_f1_macro,std_f1_macro
1,1,gini,3,1,5,0.7358,0.0814
2,1,gini,3,1,10,0.7358,0.0814
50,3,entropy,3,2,10,0.7354,0.1058
47,3,entropy,3,1,10,0.7354,0.1058
46,5,entropy,3,1,5,0.7191,0.0923
5,6,gini,3,2,10,0.7155,0.0834
3,6,gini,3,2,2,0.7155,0.0834
4,6,gini,3,2,5,0.7155,0.0834
0,9,gini,3,1,2,0.7150,0.1099
51,10,entropy,3,5,2,0.7097,0.0924


,parameter,value
0,classifier__criterion,gini
1,classifier__max_depth,3
2,classifier__min_samples_leaf,1
3,classifier__min_samples_split,5
4,best_f1_macro,0.7358
